# Etapa 2 — Obtenção e exploração dos dados

**Projeto:** Inflação, juros e câmbio no Brasil — painel com dados abertos do Banco Central.

Este notebook documenta a análise exploratória inicial: de onde vêm os dados, como são
carregados com Pandas, quais são as variáveis e seus tipos, e que problemas de qualidade
(valores ausentes, duplicidades e inconsistências) foram encontrados.

Os dados brutos ficam em `data/raw/` (um CSV por série, exatamente como a API devolve) e são
atualizados com `python scripts/atualizar_dados.py`.

In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd
import plotly.io as pio

sys.path.insert(0, str(Path.cwd().parent))
pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)

from painel_macro.config import CATALOGO, DATA_INICIAL_PADRAO, DIRETORIO_DADOS_BRUTOS
from painel_macro.fontes import FonteDadosCSV
from painel_macro.tratamento import TratadorDados

## 1. Fonte de dados

Os dados vêm do **Sistema Gerenciador de Séries Temporais (SGS)** do Banco Central do Brasil,
consultado pela API pública `https://api.bcb.gov.br/dados/serie/bcdata.sgs.{código}/dados`.
Cada série é identificada por um código numérico. O catálogo do projeto reúne 16 séries:

In [2]:
catalogo = pd.DataFrame(
    [(s.codigo, s.nome, s.rotulo, s.unidade, s.periodicidade, s.categoria) for s in CATALOGO],
    columns=["codigo_sgs", "coluna", "descricao", "unidade", "periodicidade", "categoria"],
)
catalogo

,codigo_sgs,coluna,descricao,unidade,periodicidade,categoria
0,433,ipca,IPCA mensal,% a.m.,mensal,macro
1,13522,ipca_12m,IPCA em 12 meses,% em 12 meses,mensal,macro
2,4189,selic,Taxa Selic,% a.a.,mensal,macro
3,3698,dolar,Dólar (venda),R$/US$,mensal,macro
4,24369,desocupacao,Taxa de desocupação,%,mensal,macro
5,24363,ibc_br,IBC-Br,índice,mensal,macro
6,1635,alimentacao_bebidas,Alimentação e bebidas,% a.m.,mensal,grupo_ipca
7,1636,habitacao,Habitação,% a.m.,mensal,grupo_ipca
8,1637,artigos_residencia,Artigos de residência,% a.m.,mensal,grupo_ipca
9,1638,vestuario,Vestuário,% a.m.,mensal,grupo_ipca


## 2. Carga dos dados brutos com Pandas

A classe `FonteDadosCSV` lê a cópia local das séries. Os valores chegam **como texto**, do
mesmo jeito que a API os entrega (data no formato `dd/mm/aaaa` e número com ponto decimal).

In [3]:
fonte = FonteDadosCSV(DIRETORIO_DADOS_BRUTOS)
bruto = fonte.carregar_varias(CATALOGO, DATA_INICIAL_PADRAO, date.today())
print(f"{len(bruto):,} registros brutos, {bruto['serie'].nunique()} séries")
bruto.head()

4,662 registros brutos, 16 séries


,serie,data,valor
0,ipca,01/01/2000,0.62
1,ipca,01/02/2000,0.13
2,ipca,01/03/2000,0.22
3,ipca,01/04/2000,0.42
4,ipca,01/05/2000,0.01


In [4]:
bruto.info()

<class 'pandas.DataFrame'>
RangeIndex: 4662 entries, 0 to 4661
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   serie   4662 non-null   str  
 1   data    4662 non-null   str  
 2   valor   4662 non-null   str  
dtypes: str(3)
memory usage: 223.4 KB


### Cobertura de cada série

Nem todas as séries começam em 2000 e nem todas são divulgadas com a mesma defasagem.

In [5]:
datas = pd.to_datetime(bruto["data"], format="%d/%m/%Y")
cobertura = (
    bruto.assign(data=datas)
    .groupby("serie")["data"]
    .agg(registros="count", inicio="min", fim="max")
    .sort_values("inicio")
)
cobertura

,registros,inicio,fim
serie,,,
alimentacao_bebidas,321,2000-01-01,2026-09-01
artigos_residencia,321,2000-01-01,2026-09-01
comunicacao,321,2000-01-01,2026-09-01
despesas_pessoais,321,2000-01-01,2026-09-01
dolar,322,2000-01-01,2026-10-01
educacao,321,2000-01-01,2026-09-01
habitacao,321,2000-01-01,2026-09-01
ipca,321,2000-01-01,2026-09-01
ipca_12m,321,2000-01-01,2026-09-01


## 3. Verificações de qualidade nos dados brutos

### 3.1 Valores ausentes e não numéricos

In [6]:
valores = pd.to_numeric(bruto["valor"], errors="coerce")
print("Células vazias:", int(bruto.isna().sum().sum()))
print("Valores não numéricos:", int(valores.isna().sum()))
print("Datas inválidas:", int(datas.isna().sum()))

Células vazias: 0
Valores não numéricos: 0
Datas inválidas: 0


### 3.2 Duplicidades

In [7]:
print("Linhas totalmente duplicadas:", int(bruto.duplicated().sum()))
print("Mesma série e mesma data repetidas:", int(bruto.duplicated(subset=['serie', 'data']).sum()))

Linhas totalmente duplicadas: 0
Mesma série e mesma data repetidas: 0


### 3.3 Inconsistências

Três verificações: datas no futuro, valores fora de faixas plausíveis e coerência entre séries
relacionadas.

In [8]:
hoje = pd.Timestamp.today().normalize()
print("Registros com data futura:", int((datas > hoje).sum()))

faixas = bruto.assign(valor=valores).groupby("serie")["valor"].agg(["min", "max"])
faixas

Registros com data futura: 0


,min,max
serie,,
alimentacao_bebidas,-1.070,5.850
artigos_residencia,-1.370,2.880
comunicacao,-2.080,9.310
desocupacao,5.100,14.900
despesas_pessoais,-0.310,3.550
dolar,1.564,6.097
educacao,-3.470,6.700
habitacao,-3.080,5.290
ibc_br,67.219,118.066


**Coerência entre séries.** O IPCA em 12 meses (série 13522) deve ser igual à composição
das doze últimas variações mensais do IPCA (série 433). A diferença máxima encontrada é só de
arredondamento, o que confirma que as duas séries são consistentes entre si.

In [9]:
tratador = TratadorDados()
base = tratador.tratar(bruto)

recalculado = ((1 + base["ipca"] / 100).rolling(12).apply(lambda x: x.prod(), raw=True) - 1) * 100
diferenca = (recalculado - base["ipca_12m"]).abs()
print(f"Diferença máxima entre o IPCA 12m publicado e o recalculado: {diferenca.max():.4f} p.p.")

Diferença máxima entre o IPCA 12m publicado e o recalculado: 0.0091 p.p.


## 4. Base tratada: variáveis e tipos

O `TratadorDados` converte os tipos, remove duplicidades e datas futuras, organiza tudo em uma
tabela mensal (uma linha por mês, uma coluna por série) e repete a meta anual de inflação em
todos os meses do ano.

In [10]:
base.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 323 entries, 2000-01-01 to 2026-11-01
Freq: MS
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ipca                 321 non-null    float64
 1   ipca_12m             321 non-null    float64
 2   selic                323 non-null    float64
 3   dolar                322 non-null    float64
 4   desocupacao          175 non-null    float64
 5   ibc_br               284 non-null    float64
 6   alimentacao_bebidas  321 non-null    float64
 7   habitacao            321 non-null    float64
 8   artigos_residencia   321 non-null    float64
 9   vestuario            321 non-null    float64
 10  transportes          321 non-null    float64
 11  comunicacao          321 non-null    float64
 12  saude_cuidados       321 non-null    float64
 13  despesas_pessoais    321 non-null    float64
 14  educacao             321 non-null    float64
 15  meta_inflacao        32

In [11]:
tratador.relatorio.como_tabela()

,verificacao,quantidade
0,Registros brutos recebidos,4662
1,Valores não numéricos descartados,0
2,Datas inválidas descartadas,0
3,Registros duplicados removidos,0
4,Registros com data futura removidos,0
5,Lacunas internas interpoladas,0
6,Meses na base final,323


In [12]:
base.tail(6).T

data,2026-06-01,2026-07-01,2026-08-01,2026-09-01,2026-10-01,2026-11-01
ipca,0.160,0.070,-0.320,0.820,NaN,NaN
ipca_12m,4.640,4.440,4.220,4.580,NaN,NaN
selic,14.290,14.150,13.940,13.780,13.650,13.650
dolar,5.128,5.114,5.153,5.145,5.129,NaN
desocupacao,5.400,5.300,5.300,5.200,NaN,NaN
ibc_br,110.185,114.490,111.719,NaN,NaN,NaN
alimentacao_bebidas,-0.240,-0.670,-0.340,0.830,NaN,NaN
habitacao,0.630,0.990,-1.870,2.310,NaN,NaN
artigos_residencia,0.230,0.070,0.640,0.270,NaN,NaN
vestuario,0.170,-0.660,0.120,0.710,NaN,NaN


### Valores ausentes na base tratada

Os valores ausentes que restam têm explicação e **não são preenchidos**, porque isso
significaria inventar dados:

* `desocupacao`: a PNAD Contínua só existe a partir de março de 2012;
* `ibc_br`: a série começa em janeiro de 2003 e é divulgada com cerca de dois meses de defasagem;
* demais séries: os últimos meses ainda não foram divulgados.

In [13]:
ausentes = pd.DataFrame({
    "meses_sem_valor": base.isna().sum(),
    "percentual": (base.isna().mean() * 100).round(1),
    "primeiro_mes_com_valor": base.apply(lambda coluna: coluna.first_valid_index()),
    "ultimo_mes_com_valor": base.apply(lambda coluna: coluna.last_valid_index()),
})
ausentes

,meses_sem_valor,percentual,primeiro_mes_com_valor,ultimo_mes_com_valor
ipca,2,0.600,2000-01-01,2026-09-01
ipca_12m,2,0.600,2000-01-01,2026-09-01
selic,0,0.000,2000-01-01,2026-11-01
dolar,1,0.300,2000-01-01,2026-10-01
desocupacao,148,45.800,2012-03-01,2026-09-01
ibc_br,39,12.100,2003-01-01,2026-08-01
alimentacao_bebidas,2,0.600,2000-01-01,2026-09-01
habitacao,2,0.600,2000-01-01,2026-09-01
artigos_residencia,2,0.600,2000-01-01,2026-09-01
vestuario,2,0.600,2000-01-01,2026-09-01


## 5. Análise exploratória inicial

### 5.1 Estatísticas descritivas

In [14]:
base.describe().T

,count,mean,std,min,25%,50%,75%,max
ipca,321.000,0.494,0.394,-0.680,0.250,0.450,0.670,3.020
ipca_12m,321.000,6.181,2.677,1.880,4.460,5.790,7.180,17.240
selic,323.000,12.358,4.838,1.900,9.200,12.360,14.900,26.320
dolar,322.000,3.242,1.349,1.564,2.081,2.906,4.295,6.097
desocupacao,175.000,9.521,2.866,5.100,7.000,8.600,12.100,14.900
ibc_br,284.000,94.200,10.532,67.219,89.314,96.321,101.204,118.066
alimentacao_bebidas,321.000,0.583,0.812,-1.070,0.050,0.560,1.020,5.850
habitacao,321.000,0.521,0.796,-3.080,0.220,0.470,0.770,5.290
artigos_residencia,321.000,0.290,0.542,-1.370,-0.040,0.270,0.560,2.880
vestuario,321.000,0.452,0.587,-1.440,0.090,0.480,0.850,2.110


### 5.2 Evolução das principais variáveis

In [15]:
import plotly.express as px

principais = base[["ipca_12m", "selic", "meta_inflacao"]].rename(columns={
    "ipca_12m": "IPCA em 12 meses", "selic": "Taxa Selic", "meta_inflacao": "Meta de inflação",
})
figura = px.line(principais, labels={"value": "% ao ano", "data": "", "variable": ""},
                 title="Inflação, juros e meta (2000 em diante)")
figura.update_layout(height=380, hovermode="x unified")
figura.show()

### 5.3 Distribuição da inflação mensal

In [16]:
figura = px.histogram(base, x="ipca", nbins=40,
                      labels={"ipca": "IPCA mensal (%)"},
                      title="Distribuição das variações mensais do IPCA")
figura.update_layout(height=340, yaxis_title="meses", bargap=0.05)
figura.show()

print(f"Assimetria: {base['ipca'].skew():.2f} | Meses com deflação: {(base['ipca'] < 0).sum()}")

Assimetria: 1.51 | Meses com deflação: 16


### 5.4 Correlações entre as variáveis macroeconômicas

In [17]:
base[["ipca_12m", "selic", "dolar", "desocupacao", "ibc_br"]].corr().round(2)

,ipca_12m,selic,dolar,desocupacao,ibc_br
ipca_12m,1.000,0.570,-0.030,-0.060,-0.310
selic,0.570,1.000,-0.230,-0.630,-0.470
dolar,-0.030,-0.230,1.000,0.080,0.420
desocupacao,-0.060,-0.630,0.080,1.000,-0.740
ibc_br,-0.310,-0.470,0.420,-0.740,1.000


### 5.5 Um primeiro olhar sobre os grupos do IPCA

A média mensal de fevereiro do grupo Educação é muito maior que a dos outros meses: é quando
entram os reajustes das mensalidades escolares. Esse padrão conhecido serve como mais uma
confirmação de que os códigos das séries dos grupos estão corretos.

In [18]:
educacao = base["educacao"].groupby(base.index.month).mean().round(2)
educacao.index.name = "mes"
educacao.to_frame("variacao_media_educacao").T

mes,1,2,3,4,5,6,7,8,9,10,11,12
variacao_media_educacao,0.480,4.710,0.380,0.130,0.080,0.080,0.070,0.310,0.110,0.040,0.070,0.160


## 6. Conclusões da exploração

* Os dados brutos chegam limpos: **sem valores não numéricos, sem duplicidades e sem datas
  futuras** nas séries usadas. Ainda assim, o `TratadorDados` trata esses casos, porque a API
  pode mudar e outras séries (como a meta diária da Selic) trazem datas futuras.
* O principal cuidado é com os **valores ausentes estruturais**: séries que começam depois de
  2000 e meses recentes ainda não divulgados. Eles são mantidos como ausentes.
* As séries são coerentes entre si (o IPCA em 12 meses bate com a composição do IPCA mensal).
* A inflação mensal é assimétrica à direita: a maior parte dos meses fica entre 0,2% e 0,7%,
  com poucos meses de alta muito forte.

As perguntas do projeto são respondidas no dashboard (`streamlit run app.py`) e na
documentação.